# CS 1066 Classnotes (20261116)

### As You Get Settled

1.  Start your CS1066 codespace. Go to `https://github.com/<your_username>/cs1066`, and either launch a new or restart the existing codespace for this repo.

2.  In this codespace, run `python3 grab1066.py m16a`. If you get an error, run `cd /workspaces/cs1066` in the terminal and then rerun the grab command.

3.  Use the Qwickly Attendance app to scan the QR code shown in class. Then, consider putting your cellphone away.

----
----

### Today's topics:

1.  Understand the MCP Architecture
2.  Play with the MCP Inspector on the Everything Server
3.  Why MCP Instead of Plain Tool Use?
4.  Revisiting `ring4.py` with an MCP connection
5.  Looking Forward

**NOTE:** All code in these class notes were AI-generated from my prompts. I only tested those that I could without a Claude API key.

----
----

### Musing on Model Context Protocol (MCP)

MCP was mentioned in Sid Pardeshi's talk slides, which you can find [here](https://canvas.harvard.edu/courses/173874/files).

-   E.g., Slide 16 defines MCP as "a standard plug for tool servers, like the calendar." It's used in Demo 1 for taking appointments mentioned in your messages and putting them in your calendar.

A number of you have asked me about MCP at various points in the semester. Let's try to understand this technology.

> NOTE: It's not clear to me that this technology is fundamental knowledge. I see it more as an application of good software engineering principles. You can build AI-generated prototypes without it.

----
----

### MCP Architecture and Other Key Points

MCP:

-   Is an [open-source standard](https://github.com/modelcontextprotocol).
-   [Introduced by Anthropic](https://www.anthropic.com/news/model-context-protocol) in late 2024.
-   Works with any AI model.
-   Provides a standard way for connecting AI models/agents to systems and data sources.

MCP was designed to be like REST and LSP architectural styles.

MCP is a client-server architecture.

-   Servers publish:
    - **tools** (functions) that a client can invoke to perform an action.
    - **resources**, which you can think of as read-only data _that provides context_.
    - **prompt templates**, which are written by experts. Avoids the need for you to understand and practice good prompting techniques.

-   MCP servers sit between:
    - an application/harness (containing an MCP client), and
    - some other application or system that the application wishes to interact with.

**Example:** Sid's Inbox Messages to Google Calendar workflow uses an MCP-based Google Calendar Server, which publishes the following _tools_ (see `speed-of-thought-agents/mcp_servers/gcal_server.py`):

-   list_events
-   get_event
-   create_event
-   update_event
-   delete_event

----
----

### An Everything MCP Server

In general, things look like the following, where we'll use the official MCP Inspector to explore what the example "everything" server publishes:

> Us --> MCP Inspector (client) --MCP--> `server-everything` (server)

Notes:

-   `server-everything` is a published reference server from the MCP project, which demonstrates every MCP feature.

-   The MCP Inspector is an official tool for viewing and testing MCP servers. You can use it with any MCP server.

-   Both the inspector and everything-server are written in Node.js. I'll run them locally on my MacBook Pro. I had to install:
    - Node.js following the instructions [here](https://nodejs.org/en/download).
    - `@modelcontextprotocol/inspector` (a Node.js package), which is [the MCP Inspector](https://github.com/modelcontextprotocol/inspector).
    
-   With everything installed, run `npx @modelcontextprotocol/inspector npx -y @modelcontextprotocol/server-everything` in the terminal.

Demo ... to make the world of MCP interactions and servers more concrete.

----

### Some of the Same Demo in a Codespace Using the CLI

In the terminal,

-   Install Node by running `nvm install --lts`.

-   Check installation by running `npx --version`.

Start the server and ask it one question in the MCP protocol. You'll see the server's answer as JSON. First run takes longer as it downloads packages.

-   What tools does this server make available?

In [ ]:
npx @modelcontextprotocol/inspector --cli npx -y @modelcontextprotocol/server-everything --method tools/list

-   Call the echo tool.

In [ ]:
npx @modelcontextprotocol/inspector --cli npx -y @modelcontextprotocol/server-everything --method tools/call --tool-name echo --tool-arg message="Hello from CS1066"

-   What resources does the server publish?

In [ ]:
npx @modelcontextprotocol/inspector --cli npx -y @modelcontextprotocol/server-everything --method resources/list

-   What prompt templates does the server publish?

In [ ]:
npx @modelcontextprotocol/inspector --cli npx -y @modelcontextprotocol/server-everything --method prompts/list

----
----

### Why use MCP?

1.  **Better context for your query/request.** As is typical, an LLM or an app relying on an LLM is only as good as the context that you give it. MCP makes it easier to get the context that you need, because ... (the next 3 bullets)

2.  **Fewer custom integrations.** As easy as it is to create a custom integration between your AI application and a service (e.g., GitHub or Google Calendar) using AI code generation, it is easier to rely on the single, standardized MCP protocol (assuming the API to which you want to connect has an MCP implementation).

3.  **Dynamic tool discovery.** MCP servers publish their available tools, resources, and prompts in a machine-readable way. You (or your AI) can define tool schemas and build tool functions, but why do so if they're already built and tested?

4.  **Maintenance is the vendor's responsibility.** When a vendor publishes an MCP server, it becomes their responsibility to maintain it as their API evolves.

> What's the difference between MCP servers and tool use? Level of abstraction. MCP servers provide you with a tool schema and function, which you'd otherwise have to write.

As Sid's code illustrates, anyone (you or you through an AI) can build MCP clients and servers. But today, we'll stick with published MCP servers.

----
----

### Converting ring4.py to MCP

Recall our investigation of the agentic loop. Open `../m15/ring4.py`. This program defines:

-   Multiple tools (`create_calendar_event` and `list_calendar_events`) and their input schemas.
-   In a loop (i.e., the agentic loop), it sends these tools and a message to Claude Opus 5.5. See lines 9-45, 64-70, and 72-77.
-   The loop (lines 79-111) allows the LLM to request one or more tool uses, where the loop sends the results back to the LLM in a longer context.

Let's slightly change the prompt from:

```python
    "Schedule a one-hour all-hands ..."
```

to

```python
    """
    I'm in Boston. I want to hold a one-hour planning session on Monday, March 30, 2026
    at 10am my time. What time will that be for my teammates in Tokyo and London?
    """
```

which allows us to use a published MCP server (`mcp_server_time`).

> Walk through `m16/ring6.py`.
> 
> Don't worry if you don't understand all the covered details. Just realize that we have a client-server architecture configured and running by the time we get to line 35.

----
----

### You as the LLM

`m16/ring6_by_hand.py` is `m16/ring6.py`, except that you do the work that Claude did in `ring6.py`. `ring6_by_hand.py` uses the same MCP server settings and same MCP calls as `ring6.py`.

You'll see:

-   A tool list as Claude would see it. The server's author wrote the schemas.
-   A query asking you which tool to call, and if necessary, the inputs in the tool's schema.
-   After your choice is sent to the server, the reply is printed (or an error reported).

> Feel free to look at the code in `m16/ring6_by_hand.py`. The comments highlight what's the same and different with `m16/ring6.py`.

----
----

### Looking forward

As I mentioned in my musings, what we thought we needed with MCP has been changing as models have improved (i.e., LLMs do a great job of creating clients that handle tool_use).

If you ask what's coming next, you'll learn about:

-   How recent spec updates (such as MCP 2026-07-28) have shifted MCP from a stateful, bidirectional protocol to a stateless, request/response model. [LINK](https://claude.com/resources/articles/bringing-mcp-2026-07-28-to-claude)
-   The commonly occurring confusion between skills and MCP, which should work together. MCP is how an agent calls tools while skills state what an agent should do with those tools.
-   And more.

Some related reads, which I offer without endorsement:

- "MCP Isn't Dying. It's Being Squeezed from Three Sides in 2026." [LINK](https://medium.com/@visrow/mcp-isnt-dying-it-s-being-squeezed-from-three-sides-in-2026-7017c17eb697)
- "The future of MCP: 2026 roadmap, enterprise adoption, and what comes next." [LINK](https://toloka.ai/blog/the-future-of-mcp-enterprise-adoption/)
- "6 Model Context Protocol alternatives to consider in 2026." [LINK](https://www.merge.dev/blog/model-context-protocol-alternatives)

----
----

### Commit Your Notes and Edits

Don't forget to review, commit, and push your edits so that you've updated your GitHub repo with today's work.

*   `git status` to review what's new and what's been edited.
*   `git add <list_of_files>` to indicate what you want committed.
*   `git commit -m "<your_commit_message>"` to commit the changes to the codespace's filesystem.
*   `git push` to push those committed changes to your GitHub repo.
*   `git status` to verify that you haven't forgotten anything.

Version 20261116